# Mô phỏng hệ thống cảnh báo va chạm

Notebook trình bày **cùng logic** của các module trong repository, phục vụ chạy thử và vấn đáp. Ứng dụng Streamlit chính chạy riêng bằng `streamlit run app/main.py` từ thư mục gốc. Không dùng dữ liệu cảm biến thật.


## TV2 — Bộ luật IF–THEN
Công thức: `v = speed_kmh / 3.6`; `D_stop = v*1 + v**2/(2*5)`; `D_warning = 1.5*D_stop`. R01 → DANGER, R02 → WARNING, R03 → SAFE.


In [ ]:
"""Bộ suy diễn cảnh báo va chạm bằng luật IF–THEN (mô phỏng học tập)."""

import math

REACTION_TIME_S = 1.0
DECELERATION_M_S2 = 5.0
WARNING_FACTOR = 1.5


def assess_risk(speed_kmh: float, distance_m: float) -> dict[str, str | float]:
    """Trả mức cảnh báo, mã luật, lời giải thích và ngưỡng khoảng cách.

    Input: tốc độ km/h, khoảng cách m (số hữu hạn, không âm).
    Output: level, reason, rule_id, stopping_distance_m, warning_distance_m.
    Lưu ý: đây chỉ là mô phỏng, không dùng cho phương tiện thật.
    """
    for name, value in (("speed_kmh", speed_kmh), ("distance_m", distance_m)):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise ValueError(f"{name} phải là số thực hoặc số nguyên.")
        if not math.isfinite(value) or value < 0:
            raise ValueError(f"{name} phải là số hữu hạn không âm.")

    speed_ms = speed_kmh / 3.6
    stopping_distance = (
        speed_ms * REACTION_TIME_S
        + speed_ms ** 2 / (2 * DECELERATION_M_S2)
    )
    warning_distance = WARNING_FACTOR * stopping_distance

    # Ưu tiên theo mức độ nguy hiểm: R01 > R02 > R03.
    if distance_m == 0 or distance_m <= stopping_distance:
        level = "DANGER"
        rule_id = "R01"
        reason = "Khoảng cách hiện tại không vượt quá khoảng cách dừng dự kiến."
    elif distance_m <= warning_distance:
        level = "WARNING"
        rule_id = "R02"
        reason = "Khoảng cách hiện tại nằm trong vùng cần cảnh báo."
    else:
        level = "SAFE"
        rule_id = "R03"
        reason = "Khoảng cách hiện tại vượt ngưỡng cảnh báo của mô hình."

    return {
        "level": level,
        "reason": reason,
        "rule_id": rule_id,
        "stopping_distance_m": stopping_distance,
        "warning_distance_m": warning_distance,
    }


## TV3 — Dữ liệu mô phỏng
Trong repo, `load_scenarios()` đọc file `data/scenarios.csv`. Trong notebook dùng `StringIO` chứa đúng 5 dòng CSV để có thể chạy độc lập kể cả khi mở Colab.


In [ ]:
import csv
from io import StringIO

SCENARIOS_CSV = "scenario_id,name,speed_kmh,distance_m,expected_action\nSC01,An toàn,30,30,SAFE\nSC02,Cảnh báo,30,20,WARNING\nSC03,Nguy hiểm,30,10,DANGER\nSC04,Tốc độ cao - Cảnh báo,60,50,WARNING\nSC05,Tốc độ cao - Nguy hiểm,60,20,DANGER"

def load_scenarios_notebook():
    rows = list(csv.DictReader(StringIO(SCENARIOS_CSV)))
    for row in rows:
        row["speed_kmh"] = float(row["speed_kmh"])
        row["distance_m"] = float(row["distance_m"])
    return rows

scenarios = load_scenarios_notebook()
scenarios


## TV5 — Đối chiếu kết quả
Trong repo, hàm `compare_scenarios` dùng `src.warning_engine.assess_risk`; notebook khai báo lại cùng logic so sánh và gọi trực tiếp hàm ở ô TV2 để chạy độc lập. Chỉ áp dụng cho học tập.


In [ ]:
"""Đối chiếu kết quả kỳ vọng (expected_action) với kết quả thực tế của AI (actual_level).

TV5 — issue #10. Module này KHÔNG tự viết lại luật IF–THEN: nó gọi
`assess_risk` của TV2 và chỉ làm nhiệm vụ so sánh.

Luồng dữ liệu:  CSV (TV3: load_scenarios)  ->  compare_scenarios (TV5)  ->  giao diện (TV4)
                                                    |
                                          gọi assess_risk (TV2)

Mỗi dòng kết quả có các key:
    scenario_id, name, speed_kmh, distance_m, expected_action   (lấy từ CSV)
    actual_level          SAFE/WARNING/DANGER do AI tính; None nếu ERROR
    status                PASS / FAIL / ERROR
    reason, rule_id       giải thích và mã luật từ AI ("" nếu ERROR)
    stopping_distance_m, warning_distance_m   ngưỡng AI dùng (None nếu ERROR)
    error                 nội dung lỗi ("" nếu không lỗi)

Chạy thử (từ thư mục gốc repo):  python -m src.comparison
"""

VALID_LEVELS = ("SAFE", "WARNING", "DANGER")


def _normalize_expected(value) -> str:
    """Chuẩn hóa expected_action do người nhập trong CSV: bỏ khoảng trắng, viết hoa."""
    return str(value).strip().upper() if value is not None else ""


def _compare_one(scenario: dict) -> dict:
    """So sánh một kịch bản. Mọi lỗi của riêng kịch bản này được ghi thành ERROR."""
    row = {
        "scenario_id": scenario.get("scenario_id"),
        "name": scenario.get("name"),
        "speed_kmh": scenario.get("speed_kmh"),
        "distance_m": scenario.get("distance_m"),
        "expected_action": scenario.get("expected_action"),
        "actual_level": None,
        "status": "ERROR",
        "reason": "",
        "rule_id": "",
        "stopping_distance_m": None,
        "warning_distance_m": None,
        "error": "",
    }

    expected = _normalize_expected(row["expected_action"])
    if expected not in VALID_LEVELS:
        row["error"] = f"expected_action không hợp lệ: {row['expected_action']!r}"
        return row

    try:
        result = assess_risk(row["speed_kmh"], row["distance_m"])
    except Exception as exc:  # ValueError từ TV2 hoặc dữ liệu sai kiểu
        row["error"] = f"{type(exc).__name__}: {exc}"
        return row

    # actual_level phải ĐÚNG mã chuẩn (không tự sửa chữ hoa/thường) để lộ lỗi hợp đồng của AI.
    actual = result.get("level") if isinstance(result, dict) else None
    if actual not in VALID_LEVELS:
        row["error"] = f"assess_risk trả level ngoài SAFE/WARNING/DANGER: {result!r}"
        return row

    row["actual_level"] = actual
    row["reason"] = result.get("reason", "")
    row["rule_id"] = result.get("rule_id", "")
    row["stopping_distance_m"] = result.get("stopping_distance_m")
    row["warning_distance_m"] = result.get("warning_distance_m")
    row["status"] = "PASS" if actual == expected else "FAIL"
    return row


def compare_scenarios(scenarios: list[dict]) -> list[dict]:
    """Chạy AI cho từng kịch bản và trả bảng đối chiếu.

    Mỗi dòng có tối thiểu: scenario_id, name, speed_kmh, distance_m,
    expected_action, actual_level, status (PASS / FAIL / ERROR).
    Có thêm reason, rule_id, stopping_distance_m, warning_distance_m, error
    để giao diện hiển thị lời giải thích.
    Một kịch bản lỗi không làm dừng các kịch bản còn lại.
    """
    results = []
    for scenario in scenarios:
        if not isinstance(scenario, dict):
            scenario = {}
        results.append(_compare_one(scenario))
    return results




In [ ]:
results = compare_scenarios(scenarios)
for row in results:
    print(row["scenario_id"], row["expected_action"], row["actual_level"], row["status"], row["rule_id"])
assert len(results) == 5
assert all(row["status"] == "PASS" for row in results)
print("Tổng kết: 5/5 PASS")


## TV4 — Streamlit
Giao diện sử dụng `assess_risk`, `load_scenarios`, `compare_scenarios` và `st.number_input`, `st.dataframe`. **Không chạy trực tiếp phần UI trong notebook**; toàn bộ mã nguồn giao diện gốc vẫn ở [`app/main.py`](../app/main.py). Mở VS Code và chạy `streamlit run app/main.py` để trình diễn.

## TV1 — Tích hợp
Đầu vào từ Streamlit / CSV → TV2 suy diễn → TV5 đối chiếu → TV4 hiển thị. TV1 kiểm thử luồng này và tổng hợp báo cáo.


## Mã gốc TV3 trong ứng dụng

```python
import csv
import math
from pathlib import Path

DEFAULT_CSV_PATH = Path(__file__).resolve().parent.parent / "data" / "scenarios.csv"
REQUIRED_COLUMNS = {"scenario_id", "name", "speed_kmh", "distance_m", "expected_action"}


def load_scenarios(csv_path: str | Path = DEFAULT_CSV_PATH) -> list[dict]:
    """Đọc danh sách kịch bản mô phỏng từ file CSV chuẩn 5 cột."""
    path = Path(csv_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file kịch bản: {path}")

    scenarios: list[dict] = []
    with path.open(mode="r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or not REQUIRED_COLUMNS.issubset(set(reader.fieldnames)):
            missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
            raise ValueError(f"File CSV thiếu cột bắt buộc: {sorted(missing)}")

        for row_num, row in enumerate(reader, start=2):
            try:
                speed = float(row["speed_kmh"])
                distance = float(row["distance_m"])
            except (TypeError, ValueError) as exc:
                raise ValueError(f"Dòng {row_num}: speed_kmh hoặc distance_m không phải số hợp lệ.") from exc

            if not (math.isfinite(speed) and math.isfinite(distance)) or speed < 0 or distance < 0:
                raise ValueError(f"Dòng {row_num}: speed_kmh và distance_m phải là số hữu hạn không âm.")

            scenarios.append({
                "scenario_id": (row["scenario_id"] or "").strip(),
                "name": (row["name"] or "").strip(),
                "speed_kmh": speed,
                "distance_m": distance,
                "expected_action": (row["expected_action"] or "").strip(),
            })

    return scenarios


if __name__ == "__main__":
    for item in load_scenarios():
        print(item)
```


## TV4 — Giao diện Streamlit tích hợp

Ô bên dưới chứa **toàn bộ code `app/main.py`** và xuất nó thành tệp khi chưa tồn tại. Notebook chạy thuật toán trực tiếp; để xem giao diện, cài `streamlit` và mở terminal chạy `streamlit run app/main.py` trong thư mục repository. Nếu chạy notebook riêng ngoài repo, cần có các module ở `src/` và CSV ở `data/`.


In [ ]:
from pathlib import Path

STREAMLIT_APP_SOURCE = "import sys\nfrom pathlib import Path\nimport streamlit as st\n\nROOT_DIR = Path(__file__).resolve().parent.parent\nif str(ROOT_DIR) not in sys.path:\n    sys.path.insert(0, str(ROOT_DIR))\n\nfrom src.warning_engine import assess_risk\nfrom src.simulator import load_scenarios\nfrom src.comparison import compare_scenarios\n\ndef _format_num(val):\n    if val is None:\n        return \"N/A\"\n    if isinstance(val, float):\n        return f\"{val:.2f}\"\n    return str(val)\n\ndef _render_assessment_card(speed_val, distance_val):\n    try:\n        res = assess_risk(speed_val, distance_val)\n    except ValueError as exc:\n        st.error(f\"Lỗi dữ liệu đầu vào: {exc}\")\n        return\n    except Exception as exc:\n        st.error(f\"Lỗi khi đánh giá rủi ro: {exc}\")\n        return\n\n    level = str(res.get(\"level\", \"\")).upper()\n    rule_id = res.get(\"rule_id\", \"N/A\")\n    reason = res.get(\"reason\", \"\")\n    d_stop = res.get(\"stopping_distance_m\")\n    d_warn = res.get(\"warning_distance_m\")\n\n    m1, m2, m3 = st.columns(3)\n    m1.metric(\"Tốc độ hiện tại\", f\"{float(speed_val):.1f} km/h\")\n    m2.metric(\"Khoảng cách thực tế\", f\"{float(distance_val):.1f} m\")\n    m3.metric(\"D_stop / D_warning\", f\"{_format_num(d_stop)}m / {_format_num(d_warn)}m\")\n\n    if level == \"SAFE\":\n        st.success(f\"### XANH — SAFE\\n{reason}\")\n    elif level == \"WARNING\":\n        st.warning(f\"### VÀNG — WARNING\\n{reason}\")\n    elif level == \"DANGER\":\n        st.error(f\"### ĐỎ — DANGER\\n{reason}\")\n    else:\n        st.info(f\"### {level}\\n{reason}\")\n\n    with st.container(border=True):\n        st.markdown(\"#### Giải thích kết quả & Luật được kích hoạt\")\n        st.markdown(f\"- **Mức cảnh báo:** `{level}`\")\n        st.markdown(f\"- **Mã luật:** `{rule_id}`\")\n        st.markdown(f\"- **Ngưỡng dừng (`stopping_distance_m`):** `{_format_num(d_stop)} m`\")\n        st.markdown(f\"- **Ngưỡng cảnh báo (`warning_distance_m`):** `{_format_num(d_warn)} m`\")\n        st.markdown(f\"- **Giải thích chi tiết:** {reason}\")\n\nst.set_page_config(page_title=\"Mô phỏng Hệ thống Cảnh báo Va chạm\", page_icon=\"🚗\", layout=\"wide\")\n\nst.title(\"🚗 Mô Phỏng Hệ Thống Cảnh Báo Va Chạm (FCW)\")\nst.caption(\"TV4 — Giao diện Streamlit (Issue #11)\")\n\nwith st.expander(\"Hướng dẫn sử dụng & Quy ước mức cảnh báo\", expanded=False):\n    st.markdown(\"\"\"\n- **Bước 1:** Nhập trực tiếp **Tốc độ (km/h)** và **Khoảng cách (m)** hoặc chọn kịch bản từ danh sách CSV của TV3.\n- **Bước 2:** Nhấn **Đánh giá** (hoặc **Chạy kịch bản đã chọn**) để gọi bộ luật từ `src.warning_engine`.\n- **Bước 3:** Xem mức cảnh báo (**SAFE / Xanh**, **WARNING / Vàng**, **DANGER / Đỏ**), mã luật, ngưỡng khoảng cách và bảng đối chiếu từ TV5.\n    \"\"\")\n\ntry:\n    csv_scenarios = load_scenarios()\nexcept Exception as exc:\n    st.error(f\"Không thể tải danh sách kịch bản: {exc}\")\n    csv_scenarios = []\n\ncol_input, col_output = st.columns([1, 1.6], gap=\"large\")\n\nwith col_input:\n    st.subheader(\"1. Khu vực nhập dữ liệu\")\n    with st.form(\"manual_form\"):\n        speed = st.number_input(\"Tốc độ (km/h)\", value=30.0, step=1.0, format=\"%.2f\")\n        distance = st.number_input(\"Khoảng cách (m)\", value=30.0, step=1.0, format=\"%.2f\")\n        submitted = st.form_submit_button(\"Đánh giá\", type=\"primary\", use_container_width=True)\n\n    st.divider()\n    st.subheader(\"Kịch bản từ TV3\")\n    selected_scenario = None\n    run_scenario_btn = False\n    if csv_scenarios:\n        labels = [\n            f\"{sc.get('scenario_id', '')} - {sc.get('name', '')} ({sc.get('speed_kmh')} km/h, {sc.get('distance_m')} m)\"\n            for sc in csv_scenarios\n        ]\n        sel_idx = st.selectbox(\"Chọn kịch bản:\", options=list(range(len(csv_scenarios))), format_func=lambda i: labels[i])\n        selected_scenario = csv_scenarios[sel_idx]\n        run_scenario_btn = st.button(\"Chạy kịch bản đã chọn\", use_container_width=True)\n    else:\n        st.info(\"Chưa có kịch bản nào từ CSV.\")\n\nwith col_output:\n    st.subheader(\"2. Kết quả đánh giá & Giải thích luật\")\n    if submitted:\n        _render_assessment_card(speed, distance)\n    elif run_scenario_btn and selected_scenario is not None:\n        _render_assessment_card(selected_scenario.get(\"speed_kmh\"), selected_scenario.get(\"distance_m\"))\n    else:\n        st.info(\"Nhập thông số bên trái và bấm **Đánh giá**, hoặc chọn một kịch bản để xem kết quả.\")\n\nst.divider()\nst.subheader(\"3. Bảng đối chiếu kịch bản (TV5)\")\n\nif csv_scenarios:\n    try:\n        rows = compare_scenarios(csv_scenarios)\n        st.dataframe(rows, use_container_width=True)\n        passed = sum(1 for r in rows if r.get(\"status\") == \"PASS\")\n        st.write(f\"**Tổng kết đối chiếu:** `{passed}/{len(rows)}` kịch bản đạt `PASS`.\")\n        for r in rows:\n            st_val = r.get(\"status\")\n            sid = r.get(\"scenario_id\", \"\")\n            if st_val == \"PASS\":\n                st.success(f\"{sid}: {r.get('actual_level')} ({r.get('rule_id')}) — PASS\")\n            elif st_val == \"FAIL\":\n                st.warning(f\"{sid}: kỳ vọng {r.get('expected_action')}, thực tế {r.get('actual_level')} ({r.get('rule_id')}) — FAIL\")\n            else:\n                st.error(f\"{sid}: ERROR — {r.get('error')}\")\n    except Exception as exc:\n        st.error(f\"Lỗi khi gọi compare_scenarios: {exc}\")\nelse:\n    st.info(\"Không có kịch bản để đối chiếu.\")\n"
app_path = Path('app/main.py')
if not app_path.exists():
    app_path.parent.mkdir(parents=True, exist_ok=True)
    app_path.write_text(STREAMLIT_APP_SOURCE, encoding='utf-8')
    print('Đã xuất', app_path)
else:
    print('Giữ nguyên', app_path, '(không ghi đè code TV4)')
print('Cài thư viện: pip install streamlit')
print('Mở giao diện: streamlit run app/main.py')


## Phụ lục mã nguồn nguyên bản của TV3 và TV5

Đây là hai file **nguyên văn** từ `main` để người đọc đối chiếu. Phần code thực thi phía trên đã được điều chỉnh rất ít để notebook không phụ thuộc thư mục khi chạy riêng.

### TV3 — `src/simulator.py`

```python
import csv
import math
from pathlib import Path

DEFAULT_CSV_PATH = Path(__file__).resolve().parent.parent / "data" / "scenarios.csv"
REQUIRED_COLUMNS = {"scenario_id", "name", "speed_kmh", "distance_m", "expected_action"}


def load_scenarios(csv_path: str | Path = DEFAULT_CSV_PATH) -> list[dict]:
    """Đọc danh sách kịch bản mô phỏng từ file CSV chuẩn 5 cột."""
    path = Path(csv_path)
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy file kịch bản: {path}")

    scenarios: list[dict] = []
    with path.open(mode="r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        if not reader.fieldnames or not REQUIRED_COLUMNS.issubset(set(reader.fieldnames)):
            missing = REQUIRED_COLUMNS - set(reader.fieldnames or [])
            raise ValueError(f"File CSV thiếu cột bắt buộc: {sorted(missing)}")

        for row_num, row in enumerate(reader, start=2):
            try:
                speed = float(row["speed_kmh"])
                distance = float(row["distance_m"])
            except (TypeError, ValueError) as exc:
                raise ValueError(f"Dòng {row_num}: speed_kmh hoặc distance_m không phải số hợp lệ.") from exc

            if not (math.isfinite(speed) and math.isfinite(distance)) or speed < 0 or distance < 0:
                raise ValueError(f"Dòng {row_num}: speed_kmh và distance_m phải là số hữu hạn không âm.")

            scenarios.append({
                "scenario_id": (row["scenario_id"] or "").strip(),
                "name": (row["name"] or "").strip(),
                "speed_kmh": speed,
                "distance_m": distance,
                "expected_action": (row["expected_action"] or "").strip(),
            })

    return scenarios


if __name__ == "__main__":
    for item in load_scenarios():
        print(item)
```

### TV5 — `src/comparison.py`

```python
"""Đối chiếu kết quả kỳ vọng (expected_action) với kết quả thực tế của AI (actual_level).

TV5 — issue #10. Module này KHÔNG tự viết lại luật IF–THEN: nó gọi
`assess_risk` của TV2 và chỉ làm nhiệm vụ so sánh.

Luồng dữ liệu:  CSV (TV3: load_scenarios)  ->  compare_scenarios (TV5)  ->  giao diện (TV4)
                                                    |
                                          gọi assess_risk (TV2)

Mỗi dòng kết quả có các key:
    scenario_id, name, speed_kmh, distance_m, expected_action   (lấy từ CSV)
    actual_level          SAFE/WARNING/DANGER do AI tính; None nếu ERROR
    status                PASS / FAIL / ERROR
    reason, rule_id       giải thích và mã luật từ AI ("" nếu ERROR)
    stopping_distance_m, warning_distance_m   ngưỡng AI dùng (None nếu ERROR)
    error                 nội dung lỗi ("" nếu không lỗi)

Chạy thử (từ thư mục gốc repo):  python -m src.comparison
"""

try:  # chạy từ thư mục gốc: streamlit / python -m src.comparison
    from src.warning_engine import assess_risk
except ModuleNotFoundError:  # chạy trực tiếp: python src/comparison.py
    from warning_engine import assess_risk

VALID_LEVELS = ("SAFE", "WARNING", "DANGER")


def _normalize_expected(value) -> str:
    """Chuẩn hóa expected_action do người nhập trong CSV: bỏ khoảng trắng, viết hoa."""
    return str(value).strip().upper() if value is not None else ""


def _compare_one(scenario: dict) -> dict:
    """So sánh một kịch bản. Mọi lỗi của riêng kịch bản này được ghi thành ERROR."""
    row = {
        "scenario_id": scenario.get("scenario_id"),
        "name": scenario.get("name"),
        "speed_kmh": scenario.get("speed_kmh"),
        "distance_m": scenario.get("distance_m"),
        "expected_action": scenario.get("expected_action"),
        "actual_level": None,
        "status": "ERROR",
        "reason": "",
        "rule_id": "",
        "stopping_distance_m": None,
        "warning_distance_m": None,
        "error": "",
    }

    expected = _normalize_expected(row["expected_action"])
    if expected not in VALID_LEVELS:
        row["error"] = f"expected_action không hợp lệ: {row['expected_action']!r}"
        return row

    try:
        result = assess_risk(row["speed_kmh"], row["distance_m"])
    except Exception as exc:  # ValueError từ TV2 hoặc dữ liệu sai kiểu
        row["error"] = f"{type(exc).__name__}: {exc}"
        return row

    # actual_level phải ĐÚNG mã chuẩn (không tự sửa chữ hoa/thường) để lộ lỗi hợp đồng của AI.
    actual = result.get("level") if isinstance(result, dict) else None
    if actual not in VALID_LEVELS:
        row["error"] = f"assess_risk trả level ngoài SAFE/WARNING/DANGER: {result!r}"
        return row

    row["actual_level"] = actual
    row["reason"] = result.get("reason", "")
    row["rule_id"] = result.get("rule_id", "")
    row["stopping_distance_m"] = result.get("stopping_distance_m")
    row["warning_distance_m"] = result.get("warning_distance_m")
    row["status"] = "PASS" if actual == expected else "FAIL"
    return row


def compare_scenarios(scenarios: list[dict]) -> list[dict]:
    """Chạy AI cho từng kịch bản và trả bảng đối chiếu.

    Mỗi dòng có tối thiểu: scenario_id, name, speed_kmh, distance_m,
    expected_action, actual_level, status (PASS / FAIL / ERROR).
    Có thêm reason, rule_id, stopping_distance_m, warning_distance_m, error
    để giao diện hiển thị lời giải thích.
    Một kịch bản lỗi không làm dừng các kịch bản còn lại.
    """
    results = []
    for scenario in scenarios:
        if not isinstance(scenario, dict):
            scenario = {}
        results.append(_compare_one(scenario))
    return results


if __name__ == "__main__":
    try:
        from src.simulator import load_scenarios
    except ModuleNotFoundError:
        from simulator import load_scenarios

    try:
        rows = compare_scenarios(load_scenarios())
    except (FileNotFoundError, ValueError) as exc:  # CSV thiếu hoặc sai định dạng (lỗi của TV3)
        raise SystemExit(f"Không đọc được kịch bản từ CSV: {exc}")

    print(f"{'ID':<6}{'speed':>7}{'dist':>7}  {'expected':<9}{'actual':<9}{'status':<6} rule")
    for r in rows:
        print(
            f"{str(r['scenario_id']):<6}{str(r['speed_kmh']):>7}{str(r['distance_m']):>7}  "
            f"{str(r['expected_action']):<9}{str(r['actual_level']):<9}{r['status']:<6} {r['rule_id']}"
            + (f"  <- {r['error']}" if r["error"] else "")
        )
    counts = {s: sum(r["status"] == s for r in rows) for s in ("PASS", "FAIL", "ERROR")}
    print("Tổng:", counts)

```
